# A worked example of the McEliece cryptosystem

First we create the class irreducible binary Goppa codes. We fix $q=2$ and require $n=2^m$, $1 <t <n$ as inputs.

We want to obtain $\Gamma(L, g)$ binary irreducible Goppa code with support $L= \{\gamma_0, \gamma_1, \dots, \gamma_{n-1}\} = \mathbb{F}_{2^m}$ and Goppa polynomial $g(x) \in \mathbb{F}_{2^m}[x]$ irreducible of degree $t$. The polynomial $g(x)$ can be either given as an input or randomly selected (as in the original McEliece cryptosystem). 

The parameters of $\Gamma(L, g)$ are:
- $n= 2^m$;
- $k \ge n -mt$;
- $d_{min} \ge 2t +1$, so that the error-correction capability is at least $t$.

A generator matrix, a parity-check matrix and the list of codewords are attributes of the code.

In [29]:
def hamming_weight(v):
    """
    Return the Hamming weight of a vector.

    The Hamming weight is the number of non-zero entries of the vector.

    Args:
        v: A vector.

    Returns:
        int: The Hamming weight of v.
    """
    return sum(x != 0 for x in v)

def hamming_weight_for_polynomials(p):
    """
    Return the number of nonzero coefficients of a polynomial p.

    Args:
        p: A polynomial.
    """
    return sum(x != 0 for x in p.list())


In [30]:
class BinaryGoppaCode:
    """
    Create a binary irreducible Goppa code starting from a given set of parameters. 
    
    The support is the whole extension field of F_2 and g is an irreducible polynomial of
    degree t, either provided by the user or selected randomly.
    """

    def __init__(self, n, t, g=None):
        """
        Initialise a binary Goppa code and store its parameters as attributes.

        Check that the input parameters are valid, i.e. that n= 2^m and 1 < t < n. 
        If the polynomial g is provided, check that it has degree t and is monic and 
        irreducible over F_{2^m}.
        If g not provided, randomly select an irreducible polynomial of degree t over F_{2^m}.

        Compute a generator matrix G and parity-check matrix H.

        Args:
            n(int): The desired length of the code. It must be a power of 2.
            t(int): The error-correction capability of the code and the degree of the Goppa polynomial.
            g(univariate polynomial): The irreducible Goppa polynomial. If not provided,
            one is selected randomly.

        """

        
        # Check parameters
        self.m = ZZ(n).log(2)
        if 2**self.m != n:
            raise ValueError("n must be a power of 2")
        
        if not (1< t < n):
            raise ValueError("t must be greater than 1 and less than n")

        self.n = n
        self.t = t

        # Construct F_2, F_{2^m} and polynomial ring over F_{2^m}
        self.F2 = GF(2)
        self.F = GF(2**self.m, name = 'a')
        R.<x> = PolynomialRing(self.F)

        # check g if provided, else select g
        if g is not None:
            if g not in R:
                raise ValueError("g must be a polynomial over F_{2^m}")
            
            if g.degree() != self.t:
                raise ValueError("g must have degree t")

            if not g.is_monic():
                raise ValueError("g must be monic")
                
            if not g.is_irreducible():
                raise ValueError("g must be irreducible over F_{2^m}")
            
            self.g = g
        else:
            while True: # g has to be monic and irreducible of degree t
                g = x**self.t + R.random_element(degree=self.t - 1)
                if g.is_irreducible():
                    break
            self.g = g

        #construct the support L = F_{2^m}
        self.L = list(self.F)
        
        # Construct H with coefficients in F_{2^m}
        Hbar = matrix(self.F, self.t, self.n)

        
        for j, gamma in enumerate(self.L): #the cicle constructs the matrix by columns
            inv_g = 1 / self.g(gamma)
        
            for i in range(self.t):
                Hbar[i, j] = inv_g * gamma**i

        #construct H with coefficients in F_2
        H = matrix(self.F2, self.t*self.m, self.n)

        for i in range(self.t):
            for j in range(self.n):
                binary_representation = self.binary_representation(Hbar[i, j], self.m)
        
                for r in range(self.m):
                    H[i*self.m + r, j] = binary_representation[r]

        self.H = H
        self.Hbar = Hbar

        #compute dimension of the code
        self.k = self.n - self.H.rank()

        #construct the generator matrix G
        self.G = matrix(self.F2, self.H.right_kernel().basis())
        if self.G.nrows() != self.k or self.G.ncols() != self.n:
            raise ValueError("The generator matrix does not have \
            the expected dimensions.")
        

    def codewords(self):
        """
        Return a list containing all the codewords of self.
        """
        codewords = []
        Fk = list(GF(2**self.k, name = 'b'))
        for v in Fk:
            u = self.binary_representation(v, self.k)
            codewords.append(u * self.G)
        return codewords

    def binary_representation(self, z, s):
        """
        Return the binary vector representation of an element of F_{2^s}.

        Args:
            s(int): The integer identifying the extension F_{2^s}
            z(element in the extension): An element of F_{2^s}.

        Returns:
            List: A list containing all the coefficients of z seen as a polynomial
            in a, ordered increasingly from degree 0 to the degree
            of the polynomial representation of z, plus some zeros
            so that the list has length s.
        
        """
        coefficients = z.polynomial().list() 
        coefficients += [0] * (s - len(coefficients))
        return vector(self.F2, coefficients)

    def is_in_Goppa(self, v):
        """
        Given a binary vector of dimension n, return True if it is a codeword
        of the Goppa code, False otherwise.
        """
        if len(v) != self.n:
            raise ValueError("The vector v does not have dimension n")
            
        if v.base_ring() != self.F2:
            raise ValueError("The vector v is not binary")

        return self.H * v == 0
            
    def generate_codeword(self):
        """
        Return a codeword of the Goppa code generated randomly.
        """
        u = random_vector(self.F2, self.k)

        return u * self.G

In [31]:
#Define the Goppa code of the example
m = 3
n = 2 ** m
t = 2
F2 = GF(2)
F = GF(2**m, name='a')
a = F.gen()
R.<x> = PolynomialRing(F)
g = R(x**2 + x + 1) 
Goppa = BinaryGoppaCode(n, t, g)

In [32]:
#Test the correctness of the attributes of the class
assert Goppa.n == 8
assert Goppa.m == 3
assert Goppa.t == 2
assert Goppa.F2 == F2
assert Goppa.F == F
assert Goppa.g == x^2 + x + 1
assert parent(Goppa.g) == R
assert Goppa.L == [0, a, a^2, a + 1, a^2 + a, a^2 + a + 1, a^2 + 1, 1]

assert Goppa.H.rank() == 6
assert Goppa.k == 2 
assert Goppa.H * Goppa.G.transpose() == matrix(F2, Goppa.n - Goppa.k, Goppa.k)

assert len(Goppa.codewords()) == 2**Goppa.k
assert {tuple(c) for c in Goppa.codewords()} == {(0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 1, 1, 1, 1, 1, 0), (1, 1, 1, 0, 1, 0, 0, 1), (1, 0, 0, 1, 0, 1, 1, 1)}

c = Goppa.generate_codeword()
assert Goppa.is_in_Goppa(c)

We want to simulate a McEliece cryptosystem. 

We are given a binary irreducible Goppa code $\Gamma(L, g)$, with parameters $(n = 2^m, k)$ and error-correction capability $t$. We want to randomly select
- $S \in \mathbb{F}^{k \times k}$ non singular;
- $P \in \mathbb{F}^{n \times n}$ permutation matrix.
  
Then we want to construct $G' = SGP \in  \mathbb{F}^{k \times n}$ generator matrix of the public code, which we will use for encrytpion.

In [33]:
class McEliece:
    """
    Implement the McEliece public-key cryptosystem based on a binary irreducible
    Goppa code.
    
    Given a binary Goppa code, the class randomly generates a non-singular
    scrambling matrix S and a permutation matrix P, and constructs the public
    generator matrix G' = SGP. S and P can also be given as an input.
    
    It also provides the data required for encryption
    and decryption, including Patterson's decoding algorithm. 
    """
    def __init__(self, code, S=None, P=None):
        """
        Initialise a McEliece cryptosystem from a binary Goppa code.

        Args:
            code(BinaryGoppaCode): A binary Goppa code with parameters (n, k) and 
            error-correction capability t.
        """
        if not isinstance(code, BinaryGoppaCode):
            raise ValueError("The code is not an object of the class BinaryGoppaCode")

        self.n = code.n
        self.m = code.m
        self.k = code.k
        self.t = code.t
        self.G = code.G
        self.L = code.L
        
        self.F2 = GF(2)
        self.F = GF(2**self.m, name = 'a')
        R.<x> = PolynomialRing(self.F)
        self.g = code.g

        # Randomly generate S, a non-singular matrix, and P, a permutation matrix.
        if S is not None:
            if S.base_ring() != self.F2:
                raise ValueError("S must be a matrix over F_2")
            if not (S.ncols() == self.k and S.nrows() == self.k):
                raise ValueError("The matrix S has not dimensions k x k")
            if S.is_invertible():
                self.S = S 
            else: 
                raise ValueError("The matrix S is not invertible")
        else:
            self.S = self._generate_S()

        if P is not None:
            if P.base_ring() != self.F2:
                raise ValueError("P must be a matrix over F_2")
            if not (P.ncols() == self.n and P.nrows() == self.n):
                raise ValueError("The matrix P has not dimensions n x n")
            if not all(sum(P.row(i)) == 1 for i in range(self.n)):
                raise ValueError("P is not a permutation matrix")
            if not all(sum(P.column(j)) == 1 for j in range(self.n)):
                raise ValueError("P is not a permutation matrix")
            self.P = P
        else:
            self.P = self._generate_P()

        #Compute the public generator matrix G' of the code
        self.G_public = self.S * self.G * self.P

    def _generate_S(self):
        """
        Generate a random invertible matrix of size k over F_2.
        """
        while True:
            S = random_matrix(self.F2, self.k, self.k)
            if S.is_invertible():
                return S

    def _generate_P(self):
        """
        Generate a random permutation matrix of size n over F_2.
        """
        permutation = Permutations(self.n).random_element()
        P = matrix(self.F2, self.n, self.n, sparse=True)
    
        for i, j in enumerate(permutation):
            P[i, j - 1] = 1
        return P

    def print_private_key(self):
        """
        Print the private key of the McEliece cryptosystem.
    
        The private key consists of the generator matrix G of the Goppa code,
        the Goppa polynomial g, the non-singular scrambling matrix S, and the
        permutation matrix P.
        """
        print("Private key:")
        print(f"G =\n{self.G}")
        print(f"g = {self.g}")
        print(f"S =\n{self.S}")
        print(f"P =\n{self.P}")

    def print_public_key(self):
        """
        Print the public key of the McEliece cryptosystem.
    
        The public key consists of the public generator matrix G' and the
        parameters n, k, and t of the public code.
        """
        print("Public key:")
        print(f"G' =\n{self.G_public}")
        print(f"n = {self.n}")
        print(f"k = {self.k}")
        print(f"t = {self.t}")

    def private_key(self):
        """
        Return the private key of the McEliece cryptosystem.
    
        Returns:
            tuple: A tuple containing the generator matrix G, the Goppa
                polynomial g, the scrambling matrix S, and the permutation
                matrix P.
        """
        return self.G, self.g, self.S, self.P

    def public_key(self):
        """
        Return the public key of the McEliece cryptosystem.
    
        Returns:
            tuple: A tuple containing the public generator matrix G',
                and the parameters n, k, and t.
        """
        return self.G_public, self.n, self.k, self.t
        
    def random_error(self):
        """
        Return a random error vector of dimension n and Hamming weight at most t.
        """
        while True:
            e = random_vector(self.F2, self.n)
            if hamming_weight(e) <= self.t:
                return e
        
    def encrypt(self, M):
        """
        Given a binary vector M of dimension k, compute the ciphertext obtained
        by multiplying M by the public generator matrix and adding a random
        error vector e of Hamming weight at most t.
        """
        if M.base_ring() != self.F2 or len(M) != self.k:
            raise ValueError("M must be a binary vector of dimension k")
        e = self.random_error()
        return M * self.G_public + e

    def _reduce_mod_g(self, p):
        """
        Reduce the univariate polynomial p in F[x] modulo g.

        Returns:
             Polynomial: The remainder of p modulo g, of degree less than deg(g).
        """
        R.<x> = PolynomialRing(self.F)
        
        if p.parent() != R:
            raise ValueError("The polynomial p is not in the ring F[x]")

        #Returns the remainder of the division between p and g   
        return p % self.g 
            

    def _patterson_decode(self, c):
        """
        Given a binary vector c of dimension n, obtained as c = codeword + error,
        recover the error vector of Hamming weight at most t.

        Solves the key equation S(x)sigma(x) = sigma'(x) mod g(x).

        The roots of the univariate polynomial sigma(x) identify the non-zero
        entries of the error vector. sigma(x) has degree at most t.
        """
        
        if c.base_ring() != self.F2 or len(c) != self.n:
            raise ValueError("c must be a binary vector of dimension n")
            
        #Compute the syndrome polynomial S(x)
        R = self.g.parent()
        x = R.gen()
        syndrome = 0
        
        for i, gamma in enumerate(self.L):
            if c[i] !=0:
                syndrome +=  inverse_mod(x + gamma, self.g)

        #If the syndrome polynomial is 0, no error has occurred and we return
        #the zero vector of dimension n
        if syndrome == 0:
            return vector(self.F2, self.n)

        h = self.g.xgcd(syndrome)[2]
        #h = inverse_mod(syndrome, self.g)

        #We want to find sigma such that syndrome * sigma = sigma' mod g(x)
        # sigma = alpha^2 + xbeta^2 
        
        if h == x :
            sigma = x
        else:
            d_squared = h + x
            
            K = R.quotient(self.g)
            z = R.gen()
            d = K(d_squared) ** (2**(self.m * self.t - 1))
            d = d.lift()

            #Need to solve the congruence d * beta = alpha mod g(x)
            #Extended Euclidean algorithm between g and d to find alpha and beta
            #alpha has degree <= t/2, beta has degree <= (t-1)/2

            #Initialise step 0 and 1 of the extended euclidean algorithm and check
            #if the conditions on the degrees are already satisfied

            r_0, r_1 = self.g, d
            s_0, s_1 = R(1), R(0)
            t_0, t_1 = R(0), R(1)
            
            if (r_1.degree() <= self.t // 2 and 
                t_1.degree() <= (self.t - 1) // 2):
                alpha = r_1
                beta = t_1

            while r_1 != 0: 
                q = r_0.quo_rem(r_1)[0]
                r_2 = r_0 - q * r_1
                s_2 = s_0 - q * s_1
                t_2 = t_0 - q * t_1 
            
                if (r_2.degree() <= self.t // 2 and 
                t_2.degree() <= (self.t - 1) // 2):
                    alpha = r_2
                    beta = t_2
                    break

                r_0, r_1 = r_1, r_2
                s_0, s_1 = s_1, s_2
                t_0, t_1 = t_1, t_2

        sigma = alpha ** 2 + x * beta ** 2
        error_positions = [0] * self.n

        for j, gamma in enumerate(self.L):
            if sigma(gamma) == 0:
                error_positions[j] = 1
                
        return vector(self.F2, error_positions)       

    def decrypt(self, ciphertext):
        """
        Given a binary vector ciphertext of dimension n, recover the message M,
        a binary vector of dimension k, such that C = MG' + e, where e
        is an error vector of dimension n and Hamming weight at most t.
        """
        #Check the input is correct 
        if ciphertext.base_ring() != self.F2 or len(ciphertext) != self.n:
            raise ValueError("The ciphertext must be a binary vector of dimension n")
            
        #The inverse of P permutation matrix is the transpose
        c = ciphertext * self.P.transpose() 
        error = self._patterson_decode(c)
        codeword = c - error
        
        #Solve the system uG = codeword, where u = mS
        u = self.G.solve_left(codeword)
        return u * self.S.inverse()
        

In [34]:
S = matrix(F2, [
    [1, 1],
    [0, 1] ])
P = matrix(F2, [
    [0, 1, 0, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1] ])

In [35]:
#Create the object of McEliece class with assigned S, P
mceliece = McEliece(Goppa, S, P)

#Test the functions printing the private and public keys
mceliece.print_private_key()
mceliece.print_public_key()

Private key:
G =
[1 0 0 1 0 1 1 1]
[0 1 1 1 1 1 1 0]
g = x^2 + x + 1
S =
[1 1]
[0 1]
P =
[0 1 0 0 0 0 0 0]
[1 0 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 1]
Public key:
G' =
[1 1 1 0 1 0 0 1]
[1 0 1 1 1 1 1 0]
n = 8
k = 2
t = 2


In [36]:
#Test mceliece has the correct attributes
assert mceliece.n == Goppa.n
assert mceliece.m == Goppa.m
assert mceliece.k == Goppa.k
assert mceliece.t == Goppa.t

assert mceliece.F2 == Goppa.F2
assert mceliece.F == Goppa.F

assert mceliece.G == Goppa.G
assert mceliece.g == Goppa.g

In [37]:
#Test the matrices S and P
assert mceliece.S == S
assert mceliece.S.base_ring() == F2
assert mceliece.S.nrows() == mceliece.k
assert mceliece.S.ncols() == mceliece.k
assert mceliece.S.is_invertible()

assert mceliece.P == P
assert mceliece.P.base_ring() == F2
assert mceliece.P.nrows() == mceliece.n
assert mceliece.P.ncols() == mceliece.n

assert all(sum(mceliece.P.row(i)) == 1 for i in range(mceliece.n))
assert all(sum(mceliece.P.column(j)) == 1 for j in range(mceliece.n))

In [38]:
#Test the attributes of the matrix G'
assert mceliece.G_public == mceliece.S * mceliece.G * mceliece.P
assert mceliece.G_public.nrows() == mceliece.k
assert mceliece.G_public.ncols() == mceliece.n
assert mceliece.G_public.rank() == mceliece.k

private = mceliece.private_key()

assert private[0] == mceliece.G
assert private[1] == mceliece.g
assert private[2] == mceliece.S
assert private[3] == mceliece.P

public = mceliece.public_key()

assert public[0] == mceliece.G_public
assert public[1] == mceliece.n
assert public[2] == mceliece.k
assert public[3] == mceliece.t

In [39]:
#Test the function random_error
for _ in range(100):
    e = mceliece.random_error()

    assert e.base_ring() == F2
    assert len(e) == mceliece.n
    assert hamming_weight(e) <= mceliece.t

In [40]:
message = vector(F2, [1, 0])
codeword = message * mceliece.G_public
ciphertext = mceliece.encrypt(message)
error = ciphertext - codeword

In [41]:
assert ciphertext.base_ring() == F2
assert len(ciphertext) == mceliece.n
assert hamming_weight(error) <= mceliece.t

In [42]:
assert mceliece.decrypt(ciphertext) == message